In [0]:
!git clone https://github.com/CMU-Perceptual-Computing-Lab/openpose
%cd ./openpose/models
!./getModels.sh
%cd /content

!pip install flask-ngrok

Cloning into 'openpose'...
remote: Enumerating objects: 22330, done.
remote: Total 22330 (delta 0), reused 0 (delta 0), pack-reused 22330
Receiving objects: 100% (22330/22330), 84.30 MiB | 37.88 MiB/s, done.
Resolving deltas: 100% (17536/17536), done.
/content/openpose/models
--2020-03-26 06:56:43--  http://posefs1.perception.cs.cmu.edu/OpenPose/models/pose/body_25/pose_iter_584000.caffemodel
Resolving posefs1.perception.cs.cmu.edu (posefs1.perception.cs.cmu.edu)... 128.2.176.37
Connecting to posefs1.perception.cs.cmu.edu (posefs1.perception.cs.cmu.edu)|128.2.176.37|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 104715850 (100M) [text/plain]
Saving to: ‘pose/body_25/pose_iter_584000.caffemodel’

pose_iter_584000.ca 100%[===================>]  99.86M  45.4MB/s    in 2.2s    

2020-03-26 06:56:46 (45.4 MB/s) - ‘pose/body_25/pose_iter_584000.caffemodel’ saved [104715850/104715850]

--2020-03-26 06:56:46--  http://posefs1.perception.cs.cmu.edu/OpenPose/models/pose

In [0]:
import cv2
import numpy as np
import PIL.Image
import json

from flask import Flask, request, Response
from flask_ngrok import run_with_ngrok

protoFile = "./openpose/models/pose/coco/pose_deploy_linevec.prototxt"
weightsFile = "./openpose/models/pose/coco/pose_iter_440000.caffemodel"
nPoints = 18
skeleton = [[1, 0], [1, 2], [1, 5], [2, 3], [3, 4], [5, 6], [6, 7], [1, 8], [8, 9], [9, 10], [1, 11], [11, 12], [12, 13], [0, 14], [0, 15], [14, 16], [15, 17]]

net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)

app = Flask(__name__)
run_with_ngrok(app)

@app.route('/pose', methods=['POST'])
def pose():
    image = np.array(PIL.Image.open(request.files['image']))
    imageWidth = image.shape[1]
    imageHeight = image.shape[0]

    # input image dimensions for the network
    inWidth = 368
    inHeight = 368
    inpBlob = cv2.dnn.blobFromImage(image, 1.0 / 255, (inWidth, inHeight), (0, 0, 0), swapRB=False, crop=False)

    net.setInput(inpBlob)

    output = net.forward()

    H = output.shape[2]
    W = output.shape[3]

    # Empty list to store the detected keypoints
    points = []

    for i in range(nPoints):
        # confidence map of corresponding body's part.
        probMap = output[0, i, :, :]

        # Find global maxima of the probMap.
        minVal, prob, minLoc, point = cv2.minMaxLoc(probMap)
        
        # Scale the point to fit on the original image
        x = (imageWidth * point[0]) / W
        y = (imageHeight * point[1]) / H

        if prob > 0.1: 
            # Add the point to the list if the probability is greater than the threshold
            points.append((int(x), int(y)))
        else :
            points.append(None)

    return Response(json.dumps({"skeleton" : skeleton, "points": points}), status=201, mimetype='application/json')
 
app.run()

 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


 * Running on http://127.0.0.1:5000/ (Press CTRL+C to quit)


 * Running on http://7bd04a5d.ngrok.io
 * Traffic stats available on http://127.0.0.1:4040


127.0.0.1 - - [26/Mar/2020 06:57:56] "POST /pose HTTP/1.1" 201 -
